# Clase 3 · Laboratorio — ETL completo: Saber 11 → Bodega en SQLite

**Trabajo en parejas · 90 min.**

En la pre-clase construyeron `dim_colegio` y un hecho parcial. Hoy completan la bodega:
- Agregar `dim_tiempo` y `dim_geografia`
- Actualizar `hecho_resultados` con las 3 claves de referencia
- Cargar el modelo estrella completo a SQLite
- Ejecutar 3 consultas analíticas reales

**Checkpoint del profesor a los 50 min** (después de la Tarea 3).

In [1]:
import pandas as pd
import sqlite3

CSV = "datos/saber11_muestra_500k.csv"
DB  = "datos/saber11_lab_etl.db"

df_raw = pd.read_csv(CSV, dtype=str)
df = df_raw.copy()

# Transformaciones base
cols_puntaje = ["PUNT_LECTURA_CRITICA","PUNT_MATEMATICAS","PUNT_C_NATURALES",
                "PUNT_SOCIALES_CIUDADANAS","PUNT_INGLES","PUNT_GLOBAL"]
for col in cols_puntaje:
    df[col] = pd.to_numeric(df[col], errors="coerce")

for col in ["COLE_NATURALEZA","COLE_JORNADA","COLE_CALENDARIO","COLE_BILINGUE",
            "COLE_DEPTO_UBICACION","COLE_MCPIO_UBICACION"]:
    df[col] = df[col].str.strip().str.upper()

df["PERIODO"] = pd.to_numeric(df["PERIODO"], errors="coerce").astype("Int64")

print(f"Datos listos: {len(df):,} filas × {df.shape[1]} columnas")

Datos listos: 500,000 filas × 22 columnas


## Tarea 1 (10 min) — Reconstruir dim_colegio + añadir dim_tiempo y dim_geografia

Ya construyeron `dim_colegio` en la pre-clase. Aquí la reconstruyen rápido y añaden las dos dimensiones que faltan.

| Dimensión | Columnas (además del ID) |
|---|---|
| `dim_colegio` | COLE_NATURALEZA, COLE_JORNADA, COLE_CALENDARIO, COLE_BILINGUE |
| `dim_tiempo` | PERIODO |
| `dim_geografia` | COLE_DEPTO_UBICACION, COLE_MCPIO_UBICACION |

In [2]:
# dim_colegio (igual que en pre-clase)
dim_colegio = (
    df[["COLE_NATURALEZA","COLE_JORNADA","COLE_CALENDARIO","COLE_BILINGUE"]]
    .drop_duplicates().reset_index(drop=True)
)
dim_colegio.insert(0, "colegio_id", dim_colegio.index + 1)
assert dim_colegio["colegio_id"].is_unique

# dim_tiempo — partir PERIODO en anio y quarter
# Ejemplo: 20254 → anio=2025, quarter=4
dim_tiempo = (
    df[["PERIODO"]]
    .drop_duplicates().sort_values("PERIODO").reset_index(drop=True)
)
dim_tiempo["anio"]    = dim_tiempo["PERIODO"] // 10
dim_tiempo["quarter"] = dim_tiempo["PERIODO"] % 10
dim_tiempo.insert(0, "tiempo_id", dim_tiempo.index + 1)
# Debe tener columnas: tiempo_id, PERIODO, anio, quarter
assert dim_tiempo["tiempo_id"].is_unique

# dim_geografia con geo_id (COLE_DEPTO_UBICACION + COLE_MCPIO_UBICACION)
dim_geografia = (
    df[["COLE_DEPTO_UBICACION","COLE_MCPIO_UBICACION"]]
    .drop_duplicates()
    .sort_values(["COLE_DEPTO_UBICACION","COLE_MCPIO_UBICACION"])
    .reset_index(drop=True)
)
dim_geografia.insert(0, "geo_id", dim_geografia.index + 1)
assert dim_geografia["geo_id"].is_unique

print(f"dim_colegio: {len(dim_colegio)} | dim_tiempo: {len(dim_tiempo)} | dim_geografia: {len(dim_geografia)}")
print("\ndim_tiempo:")
print(dim_tiempo)

dim_colegio: 40 | dim_tiempo: 4 | dim_geografia: 10

dim_tiempo:
   tiempo_id  PERIODO  anio  quarter
0          1    20194  2019        4
1          2    20204  2020        4
2          3    20214  2021        4
3          4    20224  2022        4


## Tarea 2 (20 min) — hecho_resultados con las 3 claves de referencia

En la pre-clase el hecho solo tenía `colegio_id`. Ahora hay que agregar `tiempo_id` y `geo_id`.

1. Une `df` con las 3 dimensiones mediante `.merge(..., how='left')`.
2. El hecho final debe tener: `colegio_id`, `tiempo_id`, `geo_id` + los 6 puntajes.
3. **Validación:** `len(hecho_resultados) == len(df)` — si falla, hay un join mal configurado.

In [3]:
# Unir con las 3 dimensiones para obtener las claves de referencia
df_h = df.merge(dim_colegio,
                on=["COLE_NATURALEZA","COLE_JORNADA","COLE_CALENDARIO","COLE_BILINGUE"],
                how="left")

# merge con dim_tiempo (on="PERIODO")
df_h = df_h.merge(dim_tiempo[["tiempo_id","PERIODO"]], on="PERIODO", how="left")

# merge con dim_geografia (on=["COLE_DEPTO_UBICACION","COLE_MCPIO_UBICACION"])
df_h = df_h.merge(dim_geografia,
                  on=["COLE_DEPTO_UBICACION","COLE_MCPIO_UBICACION"],
                  how="left")

cols_puntaje = ["PUNT_LECTURA_CRITICA","PUNT_MATEMATICAS","PUNT_C_NATURALES",
                "PUNT_SOCIALES_CIUDADANAS","PUNT_INGLES","PUNT_GLOBAL"]
hecho_resultados = df_h[["colegio_id","tiempo_id","geo_id"] + cols_puntaje].copy()

assert len(hecho_resultados) == len(df), f"Perdimos {len(df)-len(hecho_resultados)} filas"
# Ninguna FK puede quedar nula: si pasa, la dimensión no cubre todas las combinaciones
nulos_fk = hecho_resultados[["colegio_id","tiempo_id","geo_id"]].isna().sum()
assert nulos_fk.sum() == 0, f"FKs nulas:\n{nulos_fk}"

print(f"hecho_resultados: {len(hecho_resultados):,} filas con 3 FKs ✓")
hecho_resultados.head(3)

hecho_resultados: 500,000 filas con 3 FKs ✓


,colegio_id,tiempo_id,geo_id,PUNT_LECTURA_CRITICA,PUNT_MATEMATICAS,PUNT_C_NATURALES,PUNT_SOCIALES_CIUDADANAS,PUNT_INGLES,PUNT_GLOBAL
0,1,1,2,48,51,26,14,67,194
1,1,4,4,77,32,64,48,48,199
2,2,3,3,65,43,63,84,48,268


## Tarea 3 (15 min) — Cargar el modelo estrella completo a SQLite

Carga las 4 tablas a la BD. Luego verifica contando las filas de cada tabla.

In [4]:
conn = sqlite3.connect(DB)

# Cargar las 4 tablas con to_sql (if_exists='replace')
dim_colegio.to_sql("dim_colegio", conn, if_exists="replace", index=False)
dim_tiempo.to_sql("dim_tiempo", conn, if_exists="replace", index=False)
dim_geografia.to_sql("dim_geografia", conn, if_exists="replace", index=False)
hecho_resultados.to_sql("hecho_resultados", conn, if_exists="replace", index=False)
conn.commit()

# Verificar
for tabla in ["dim_colegio", "dim_tiempo", "dim_geografia", "hecho_resultados"]:
    n = conn.execute(f"SELECT COUNT(*) FROM {tabla}").fetchone()[0]
    print(f"  {tabla}: {n:,} filas")

  dim_colegio: 40 filas
  dim_tiempo: 4 filas
  dim_geografia: 10 filas
  hecho_resultados: 500,000 filas


## ⏸ Checkpoint del profesor (10 min)

Revisamos juntos:
- ¿Alguien perdió filas en la Tarea 2? Diagnóstico rápido.
- ¿Por qué las bodegas de datos NO imponen restricciones de FK en la base de datos?
- ¿Qué representa una fila del hecho_resultados? ¿Un estudiante, un colegio, un período?

## Tarea 4 (30 min) — Diagrama del modelo estrella + Consultas SQL

### Parte A — Diagrama del modelo (5 min)

Aquí está el modelo que construyeron. Añadan en el Markdown de abajo los atributos de cada tabla:

```
                       dim_tiempo
                       (tiempo_id  PK,
                        PERIODO,
                        anio,
                        quarter)
                            ↑
                            | tiempo_id
                            |
    dim_colegio  ←——— hecho_resultados ———→  dim_geografia
   (colegio_id PK,   (colegio_id  ref. → dim_colegio,   (geo_id PK,
    COLE_NATURALEZA,  tiempo_id   ref. → dim_tiempo,     COLE_DEPTO_UBICACION,
    COLE_JORNADA,     geo_id      ref. → dim_geografia,  COLE_MCPIO_UBICACION)
    COLE_CALENDARIO,  PUNT_LECTURA_CRITICA,
    COLE_BILINGUE)    PUNT_MATEMATICAS,
                      PUNT_C_NATURALES,
                      PUNT_SOCIALES_CIUDADANAS,
                      PUNT_INGLES,
                      PUNT_GLOBAL)
```

**Grano del hecho:** una fila = **un estudiante que presentó el examen** en un período,
en un colegio de cierto tipo y ubicación. Las medidas (los 6 puntajes) son aditivas por
promedio, no por suma.

In [5]:
# Consulta 1: promedio de PUNT_GLOBAL por naturaleza del colegio (oficial vs no oficial)
q1 = """
SELECT c.COLE_NATURALEZA,
       ROUND(AVG(h.PUNT_GLOBAL), 1) AS prom_global,
       COUNT(*) AS n_estudiantes
FROM hecho_resultados h
JOIN dim_colegio c ON h.colegio_id = c.colegio_id
GROUP BY c.COLE_NATURALEZA
ORDER BY prom_global DESC
"""
r1 = pd.read_sql(q1, conn)
print("Consulta 1:")
print(r1)

Consulta 1:
  COLE_NATURALEZA  prom_global  n_estudiantes
0      NO OFICIAL        248.8         140091
1         OFICIAL        248.6         359909


In [6]:
# Consulta 2: top 5 departamentos con mayor promedio de PUNT_MATEMATICAS
q2 = """
SELECT g.COLE_DEPTO_UBICACION AS departamento,
       ROUND(AVG(h.PUNT_MATEMATICAS), 1) AS prom_matematicas,
       COUNT(*) AS n_estudiantes
FROM hecho_resultados h
JOIN dim_geografia g ON h.geo_id = g.geo_id
GROUP BY g.COLE_DEPTO_UBICACION
ORDER BY prom_matematicas DESC
LIMIT 5
"""
r2 = pd.read_sql(q2, conn)
print("Consulta 2:")
print(r2)

Consulta 2:
      departamento  prom_matematicas  n_estudiantes
0        SANTANDER              48.4          49747
1      BOGOTÁ D.C.              48.4          50115
2        ATLÁNTICO              48.4          49692
3  VALLE DEL CAUCA              48.3          50143
4           NARIÑO              48.3          50088


In [7]:
# Consulta 3: por año y quarter, ¿en qué período mejoró más el puntaje?
q3 = """
SELECT t.anio,
       t.quarter,
       ROUND(AVG(h.PUNT_GLOBAL), 1) AS prom_global,
       COUNT(*) AS n_estudiantes
FROM hecho_resultados h
JOIN dim_tiempo t ON h.tiempo_id = t.tiempo_id
GROUP BY t.anio, t.quarter
ORDER BY t.anio ASC, t.quarter ASC
"""
r3 = pd.read_sql(q3, conn)
print("Consulta 3:")
print(r3)

# Consulta extra (para la conclusión): promedio por jornada
q4 = """
SELECT c.COLE_JORNADA,
       ROUND(AVG(h.PUNT_GLOBAL), 1) AS prom_global,
       COUNT(*) AS n_estudiantes
FROM hecho_resultados h
JOIN dim_colegio c ON h.colegio_id = c.colegio_id
GROUP BY c.COLE_JORNADA
ORDER BY prom_global DESC
"""
r4 = pd.read_sql(q4, conn)
print("\nPromedio por jornada:")
print(r4)

conn.close()

Consulta 3:
   anio  quarter  prom_global  n_estudiantes
0  2019        4        248.9         125272
1  2020        4        248.6         124499
2  2021        4        248.7         125318
3  2022        4        248.4         124911



Promedio por jornada:
  COLE_JORNADA  prom_global  n_estudiantes
0        TARDE        249.0         125323
1        NOCHE        248.6          30070
2     COMPLETA        248.6         149353
3     SABATINA        248.5          19928
4       MAÑANA        248.5         175326


### Conclusión

Mirando los resultados de las 3 consultas, escribe 3-4 frases:
- ¿Qué brecha de puntaje hay entre colegios oficiales y no oficiales?
- ¿La jornada con mejor puntaje coincide con lo que esperabas?

_Respuesta:_

La brecha entre colegios oficiales y no oficiales es prácticamente **nula**: 248.8 vs 248.6
de `PUNT_GLOBAL`, es decir **0.2 puntos** a favor de los no oficiales sobre una escala de 0 a 500.
Con una desviación estándar de ~54 puntos y medio millón de registros, esa diferencia cae dentro
del margen de error (IC 95%: ±0.33), así que no hay evidencia de brecha alguna.

Por jornada pasa lo mismo: TARDE encabeza con 249.0 y MAÑANA cierra con 248.5, apenas 0.5 puntos
entre la primera y la última de las cinco jornadas. **No** coincide con lo esperado: en los datos
reales del ICFES la jornada COMPLETA y los colegios no oficiales sacan ventajas de varias decenas
de puntos, y aquí no aparece ninguna.

Lo mismo ocurre por departamento (todos entre 48.3 y 48.4 en matemáticas) y por año
(248.4–248.9 entre 2019 y 2022, sin tendencia). Que **todos** los cortes den el mismo promedio,
sin importar la dimensión, indica que esta muestra de 500k es sintética: los puntajes se generaron
al azar, independientes de los atributos del colegio, la geografía y el período.

La conclusión metodológica es la que importa: la bodega funciona —el modelo estrella responde
las tres preguntas con un JOIN cada una— pero un modelo bien construido no inventa señal donde
no la hay. Sobre estos datos, lo correcto es reportar que no se detectan diferencias.

---

## Reflexión final — ¿Qué dimensión faltó?

Revisa las columnas del CSV de Saber 11 que **no usaste** en ninguna dimensión:

```
ESTU_GENERO, ESTU_FECHANACIMIENTO
FAMI_ESTRATOVIVIENDA, FAMI_TIENEINTERNET
FAMI_EDUCACIONMADRE, FAMI_EDUCACIONPADRE
DESEMP_INGLES
```

**Preguntas:**
1. ¿A qué dimensión pertenecen estas columnas? ¿Cómo la llamarías?
2. ¿Cuáles columnas incluirías en esa dimensión y cuáles dejarías fuera? ¿Por qué?
3. Bosqueja el código para construirla (solo la estructura — no es necesario ejecutarla):

```python
# dim_??? = (
#     df[["...", "...", "..."]]
#     .drop_duplicates()
#     .reset_index(drop=True)
# )
# dim_???.insert(0, "???_id", dim_???.index + 1)
```

_Tus respuestas:_

**1.** Describen al estudiante y a su hogar, no al colegio ni al lugar ni al tiempo.
Faltó una dimensión de perfil socio-demográfico: la llamaría **`dim_estudiante`**
(o, si se quiere separar lo del hogar, `dim_estudiante` + `dim_familia`).

**2.** Incluiría los atributos **categóricos y de baja cardinalidad**, que son los que
sirven para agrupar: `ESTU_GENERO`, `FAMI_ESTRATOVIVIENDA`, `FAMI_TIENEINTERNET`,
`FAMI_EDUCACIONMADRE`, `FAMI_EDUCACIONPADRE`. Con esas 5 columnas la dimensión queda en
unos pocos miles de combinaciones, que es el tamaño que debe tener una dimensión.

Dejaría fuera:
- `ESTU_FECHANACIMIENTO`: es casi única por estudiante, así que haría explotar la
  dimensión a ~500k filas (una dimensión del mismo tamaño que el hecho no sirve de nada).
  Lo correcto es derivarla a `edad` o `rango_edad` y meter esa versión agrupada.
- `DESEMP_INGLES`: no describe al estudiante, es el **resultado** del examen — es el
  nivel de desempeño derivado de `PUNT_INGLES`. Va en el hecho, junto a los puntajes.

**3.** Bosquejo:

```python
cols_estudiante = ["ESTU_GENERO", "FAMI_ESTRATOVIVIENDA", "FAMI_TIENEINTERNET",
                   "FAMI_EDUCACIONMADRE", "FAMI_EDUCACIONPADRE"]

dim_estudiante = (
    df[cols_estudiante]
    .drop_duplicates()
    .reset_index(drop=True)
)
dim_estudiante.insert(0, "estudiante_id", dim_estudiante.index + 1)

# y en el hecho:
# df_h = df_h.merge(dim_estudiante, on=cols_estudiante, how="left")
# hecho_resultados = df_h[["colegio_id","tiempo_id","geo_id","estudiante_id"] + cols_puntaje]
```

## Entrega

- Suban este notebook completado a Moodle antes de las 23:59.
- Nombre: `apellido1_apellido2_clase03_lab.ipynb`.